In [117]:
import pandas as pd
import shap
import joblib
import time
import json
import numpy as np
import os
from datetime import datetime

# churnprediction = pd.read_csv(r"D:\CongTyObiuty\2. DuAn\3. ThayCong\2. Obiuty\2. AiData\Database\needprediction\ai_churnprediction_25-12-12.csv")

CSV_DIR = r"D:\CongTyObiuty\2. DuAn\3. ThayCong\2. Obiuty\2. AiData\Database\needprediction"

latest_csv = max(
    [os.path.join(CSV_DIR, f) for f in os.listdir(CSV_DIR) if f.endswith(".csv")],
    key=os.path.getmtime
)

churnprediction = pd.read_csv(latest_csv)

In [118]:
# MODEL_DIR = r"D:\CongTyObiuty\2. DuAn\3. ThayCong\2. Obiuty\2. AiData\Database\training\models"

# model = joblib.load(
#     os.path.join(MODEL_DIR, "XgbChurn_v1.0_20251214.pkl")
# )
# model_xgb = model["model"]

# Lấy file mới nhất để dự đoán
import os
import joblib

MODEL_DIR = r"D:\CongTyObiuty\2. DuAn\3. ThayCong\2. Obiuty\2. AiData\Database\training\models"

model = joblib.load(
    max(
        [os.path.join(MODEL_DIR, f) for f in os.listdir(MODEL_DIR) if f.endswith(".pkl")],
        key=os.path.getmtime
    )
)

model_xgb = model["model"]


In [119]:
id_new = churnprediction["id"]
X = churnprediction.drop(columns=["spbiid","id", "window", "requesttime", "spoid", "completedtime"])

In [120]:
churn_predict = model_xgb.predict_proba(X)[:, 1]

df_pred = pd.DataFrame({
    "id": id_new.values,
    "churn_proba": churn_predict
})

# Risk score mapping (KHÔNG phá model)
df_pred["risk_score"] = 0.2 + 0.8 * df_pred["churn_proba"]
# df_pred["risk_score"] = df_pred["risk_score"].clip(0.2, 1)
df_pred_new = df_pred.sort_values(
    "risk_score", ascending=False
).reset_index(drop=True)


# df_pred = pd.DataFrame({
#     "id": id_new.values,
#     "churn_proba": churn_predict
# })

# THRESHOLD = 0.5  
# churn_pred = (churn_predict > THRESHOLD).astype(int)

# df_pred_new = df_pred.sort_values(
#     "churn_proba",
#     ascending=False
# ).reset_index(drop=True)

# # X["churn_probability"] = churn_predict
# # X["churn_prediction"] = churn_pred

In [121]:
df_pred

,id,churn_proba,risk_score
0,1097,0.009976,0.207981
1,1098,0.292609,0.434087
2,1099,0.283781,0.427025
3,1100,0.016254,0.213003
4,1101,0.006153,0.204923
...,...,...,...
58,1155,0.148139,0.318511
59,1156,0.199296,0.359437
60,1157,0.026438,0.221150
61,1158,0.047015,0.237612


In [122]:

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

explainer = shap.TreeExplainer(model_xgb)
base_value_log_odds = explainer.expected_value
base_value_prob = sigmoid(base_value_log_odds)
shap_values = explainer.shap_values(X)


In [123]:

# MODEL_CODE = "XgbChurn_V1.0"
# model_training_time = int(time.mktime(time.strptime("2025-12-12", "%Y-%m-%d")))
# predict_time = int(time.mktime(time.strptime("2025-12-16", "%Y-%m-%d")))

MODEL_CODE = model["model_version"]
model_training_time = model["train_date"]
predict_time = int(time.time())


In [124]:
today_str = datetime.now().strftime("%Y-%m-%d")
output_path = rf"D:\CongTyObiuty\2. DuAn\3. ThayCong\2. Obiuty\2. AiData\Database\logs_result\ai_churnresult2_{today_str}.csv"

ID_COL = "chrs_id"

df_insert = df_pred_new.copy()

predict_time = int(time.time())

df_insert["predict_time"] = predict_time
df_insert["churn_proba"] = df_insert["churn_proba"].round(5)
df_insert["chrs_chreid"] = df_insert["id"]
df_insert["chrs_modelcode"] = MODEL_CODE
df_insert["chrs_modeltrainingtime"] = model_training_time
df_insert["chrs_excpredicttime"] = predict_time
# df_insert["chrs_predictionvalue"] = df_insert["churn_proba"]
df_insert["chrs_predictionvalue"] = df_insert["risk_score"].round(5)


# df_insert["chrs_predictionresult"] = df_insert["churn_proba"].apply(
#     lambda x: json.dumps({"predictionresult": float(x), "label": int(x > THRESHOLD)})
# )


# TOP_K = 3

# def get_top_shap_with_percent(row):
#     shap_row = shap_values[row.name]
#     shap_dict = dict(zip(X.columns, shap_row))

#     # ===== Nhóm tăng churn =====
#     pos_items = {k: v for k, v in shap_dict.items() if v > 0}
#     pos_total = sum(pos_items.values())

#     pos_result = {}
#     if pos_total > 0:
#         pos_sorted = sorted(pos_items.items(), key=lambda x: x[1], reverse=True)[:TOP_K]
#         for k, v in pos_sorted:
#             pos_result[k] = round(float(v / pos_total), 4)
#                 # "shap_value": round(float(v), 5),
                
            

#     # ===== Nhóm giảm churn =====
#     neg_items = {k: v for k, v in shap_dict.items() if v < 0}
#     neg_total = sum(abs(v) for v in neg_items.values())

#     neg_result = {}
#     if neg_total > 0:
#         neg_sorted = sorted(neg_items.items(), key=lambda x: x[1])[:TOP_K]
#         for k, v in neg_sorted:
#             neg_result[k] = round(float(abs(v) / neg_total), 4)
#                 # "shap_value": round(float(v), 5),
                
            

#     return pos_result, neg_result

# df_insert["chrs_predictionresult"] = df_insert.apply(
#     lambda row: json.dumps(
#         {
#             "base_value_log_odds": round(float(base_value_log_odds), 3),
#             "base_value_probability": round(float(base_value_prob), 3),
#             "predictionresult": round(float(row["churn_proba"]), 5),

#             # SHAP theo %
#             "shap_increase_churn": get_top_shap_with_percent(row)[0],
#             "shap_decrease_churn": get_top_shap_with_percent(row)[1]
#         },
#         ensure_ascii=False
#     ),
#     axis=1
# )

TOP_K = 3

def get_shap_percent_all(row):
    shap_row = shap_values[row.name]
    shap_dict = dict(zip(X.columns, shap_row))

    total_abs_shap = sum(abs(v) for v in shap_dict.values())
    if total_abs_shap == 0:
        return {}, {}

    increase = {}
    decrease = {}

    for k, v in shap_dict.items():
        percent = abs(v) / total_abs_shap
        if v > 0:
            increase[k] = round(float(percent), 4)
        elif v < 0:
            decrease[k] = round(float(percent), 4)

    # TOP K theo % đóng góp
    increase = dict(sorted(increase.items(), key=lambda x: x[1], reverse=True)[:TOP_K])
    decrease = dict(sorted(decrease.items(), key=lambda x: x[1], reverse=True)[:TOP_K])

    return increase, decrease

df_insert["chrs_predictionresult"] = df_insert.apply(
    lambda row: json.dumps(
        {
            "base_value_log_odds": round(float(base_value_log_odds), 3),
            "base_value_probability": round(float(base_value_prob), 3),
            "predictionresult": round(float(row["churn_proba"]), 5),

            "risk_score": round(float(row["risk_score"]), 4),

            "shap_increase_churn": get_shap_percent_all(row)[0],
            "shap_decrease_churn": get_shap_percent_all(row)[1]
        },
        ensure_ascii=False
    ),
    axis=1
)


df_insert["chrs_status"] = 0


C:\Users\nguye\AppData\Local\Temp\ipykernel_9944\3554595749.py:104: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "base_value_log_odds": round(float(base_value_log_odds), 3),
C:\Users\nguye\AppData\Local\Temp\ipykernel_9944\3554595749.py:105: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "base_value_probability": round(float(base_value_prob), 3),


In [125]:
df_insert = df_insert[[
    "chrs_chreid",
    "chrs_modelcode",
    "chrs_modeltrainingtime",
    "chrs_excpredicttime",
    "chrs_predictionvalue",
    "chrs_predictionresult",
    "chrs_status"
]]


In [126]:
if os.path.exists(output_path):
    df_old = pd.read_csv(output_path)
    start_id = int(df_old[ID_COL].max()) + 1
    write_header = False
else:
    start_id = 1
    write_header = True

df_insert.insert(
    0,
    ID_COL,
    range(start_id, start_id + len(df_insert))
)

In [127]:
write_header = not os.path.exists(output_path)

df_insert.to_csv(
    output_path,
    mode="a",
    header=write_header,
    index=False,
    encoding="utf-8-sig"
)